# NPC external validation — spot-level check of UNI2 A8 (2026.11.16)

This notebook replaces the comparison cells of `Pred_visual_npc_uni2_A8.ipynb` (the nuclei / pseudo-spot / cell2location cell and the abundance-map cell). Those cells had four problems:

1. **cell2location fractions were spot-argmax fractions.** `cell2location 0.18 B` meant "B is the largest type in 18% of spots", not "18% of cells are B". Here the c2l cell fraction is the sum of abundances over spots.
2. **Spearman compared a proportion with a count.** Hist2Pheno mean softmax (0–1 per spot) was ranked against c2l expected cell counts (which also scale with spot density). Here proportion is compared with proportion, and count with count.
3. **DCIS was mapped to `normal`.** DCIS is neoplastic; on NPC it is a tumour-like call. `npc_figures.L2_TO_C2L["DCIS"]` is now `tumor`.
4. **Capture radius used 65 µm.** Space Ranger `spot_diameter_fullres` is a 65 µm spot; the capture area is 55 µm. Here the radius is `spot_diameter_fullres × 55/65 / 2`.

New reference: when `filtered_feature_bc_matrix.h5` is in the Visium folder, spot marker-gene scores (MS4A1/CD79A for B, MZB1/JCHAIN for plasma, CD3D/CD3E for T, ...) are used as RNA ground truth for every Hist2Pheno compartment, independent of cell2location.

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42  # TrueType; PDF text stays editable
mpl.rcParams["ps.fonttype"] = 42

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
# 2026.10.28, explicit checkout override / Farm 可设 HIST2PHENO_REPO
import os
if os.environ.get("HIST2PHENO_REPO"):
    REPO = Path(os.environ["HIST2PHENO_REPO"]).expanduser().resolve()
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code"))
sys.path.insert(0, str(REPO / "code" / "External_val" / "NPC"))

from Hist2Pheno_pkg.external_validation import ExternalCohort
from npc_figures import (
    C2L_TYPES,
    STACK_ORDER,
    c2l_palette,
    configure_figure_dpi,
    draw_spot_circles,
    draw_tls,
    group_fraction,
    keep_leftmost_foci,
    legend_handles,
    load_tls_xy,
    map_l2,
    nuclei_in_contours,
    save_figure,
    tls_score_row,
    tls_smooth_contours,
)

from Hist2Pheno_pkg.prediction_confidence import (
    plot_bcell_presence_figure,
    plot_class_argmax_vs_gated,
    plot_trusted_spatial,
)
OOF_NPZ = (
    REPO
    / "data/PanCancer/result_uni2_A8_l12_balanced_paired_v2_seed42"
    / "oof_predictions.npz"
)

CKPT = (
    REPO
    / "data/PanCancer/result_uni2_A8_l12_balanced_paired_v2_seed42"
    / "cross_dataset_cv/A8_l12_balanced/best_mlp_gpu.pt"
)


STAR_PROB_MIN = 0.00  # valley between the prob=0.01 tail and the nucleus mode


import os as _prob_os
_prob_os.environ["STAR_PROB_MIN"] = "0" if float(STAR_PROB_MIN) <= 0 else f"{float(STAR_PROB_MIN):.2f}"
# 2026.10.28, one prediction tree per cutoff / 0.40 仍写原目录，其它阈值另存
_pred_base = "pred_uni2_A8_bestmlp_paired_v2_seed42"
pred_dirname = _pred_base if float(STAR_PROB_MIN) <= 0 else f"{_pred_base}_prob{float(STAR_PROB_MIN):.2f}"
_cohort = ExternalCohort(
    name="NPC",
    data_root=REPO / "data" / "Visium" / "NPC",
    repo=REPO,
    patch_size=14,
    method="UNI2",
    embedding_dirname="project_all_UNI2",
    pred_dirname=pred_dirname,
    checkpoint=CKPT,
    project_tag="project_all_UNI2",
)
PRED_ROOT = _cohort.pred_root
composition_table = _cohort.composition_table
discover_samples = _cohort.discover_samples
geometry = _cohort.geometry
sample_paths = _cohort.sample_paths
ensure_l2_probs = _cohort.ensure_l2_probs
plot_spatial = _cohort.plot_spatial
plot_composition = _cohort.plot_composition
# Re-run the plot cell after changing these. Notebook display uses DPI; PNG uses SAVE_DPI; PDF is Type 42 vector.
SAVE_FIGURES = True
FIGSIZE = (17.2, 6.6)
DPI = 200
SAVE_DPI = 300
configure_figure_dpi(save_dpi=SAVE_DPI, plot_dpi=DPI)
HE_MAX_SIDE = 1400
MARKER_PX = 1.2
samples = discover_samples()
samples = [s for s in samples if s == "20210809-C-AH4199551"]
print("notebook shows patient1 only:", samples)
if not samples:
    raise RuntimeError("patient1 20210809-C-AH4199551 was not discovered")
print(f"StarDist probability >= {STAR_PROB_MIN:.2f}")
print(f"checkpoint {CKPT.name} exists={CKPT.is_file()}")
print(f"pred root {PRED_ROOT}")
for sample in samples:
    g = geometry(sample)
    print(
        f"{sample}  mpp={g['mpp_um']:.4f} µm/px  "
        f"native14={g['patch_um_if_native_16px']:.2f} µm  "
        f"after rescale={g['patch_um_after_rescale']:.1f} µm  "
        f"scale={g['scale']:.4f}"
    )
    p = sample_paths(sample)
    print(f"  embeddings: {p['emb']}")
    print(f"  pred exists: {p['pred'].is_file()}  {p['pred']}")

# 2026.10.28, mandatory current-input preflight / 后续直接读 h5ad 的图表也不得绕过上游
import os as _pipeline_os
_pipeline_os.environ["STAR_PROB_MIN"] = str(globals().get("STAR_PROB_MIN", 0.0))
if "ensure_l2_probs" in globals():
    _owner = ensure_l2_probs.__self__
    _current_samples = list(samples) if "samples" in globals() else _owner.discover_samples()
    _owner.cmd_predict(_current_samples, force=False, allow_cpu=True)
elif "ue" in globals():
    ue.cmd_predict(COHORT, samples, cuda_device=_pipeline_os.environ.get("CUDA_DEVICE", "0"),
                   force=False, allow_cpu=True, embedder=EMBEDDER)
else:
    raise RuntimeError("No current prediction validator; run the setup cell first")

In [ ]:
# 2026.11.16, spot-level validation (proportion vs proportion, count vs count, RNA markers)
import importlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

for _p in (REPO / "code" / "Hist2Pheno_pkg", REPO / "code" / "External_val" / "NPC"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
import visium_validation as vv
import npc_figures
importlib.reload(vv)
importlib.reload(npc_figures)
from npc_figures import C2L_TYPES, L2_TO_C2L, load_aligned_l2_probs

NPC_DATA = REPO / "data" / "Visium" / "NPC"
SAMPLE_TO_VISIUM = {"20210809-C-AH4199551": "patient1", "20210809-D-AJ3037946": "patient2"}
SAMPLE_TO_C2L = {"20210809-C-AH4199551": "AH4199551", "20210809-D-AJ3037946": "AJ3037946"}
C2L_FILES = [NPC_DATA / "Annotation" / "sp.obs.csv", NPC_DATA / "Annotation" / "AJ_sp.obs.csv"]
c2l_obs = pd.concat([pd.read_csv(p, index_col=0) for p in C2L_FILES if p.is_file()], axis=0)
c2l_obs.index = c2l_obs.index.astype(str)
fig_dir = PRED_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Hist2Pheno compartments compared with the RNA marker sets
MARKER_PAIRS = {f"frac_{g}": g for g in ("B", "Plasma", "T", "Myeloid", "Stroma", "Endothelial", "Epithelial")}
MARKER_PAIRS["frac_Tumor"] = "Tumor"
MARKER_SETS = dict(vv.MARKERS, Tumor=vv.TUMOR_MARKERS["NPC"])

results = {}
for sample in samples:
    visium_dir = NPC_DATA / SAMPLE_TO_VISIUM[sample]
    spots = vv.read_tissue_positions(visium_dir)
    radius = vv.capture_radius_px(visium_dir / "scalefactors_json.json")
    ensure_l2_probs(sample)
    pred_path = sample_paths(sample)["pred"]
    xy, probs, class_names = load_aligned_l2_probs(pred_path, pred_path.with_name(f"{sample}_l2_probs.npz"))

    # (a) cell2location groups
    h2p_c2l = vv.spot_h2p(xy, probs, class_names, spots, radius, mapping=L2_TO_C2L, groups=C2L_TYPES)
    sub = c2l_obs.loc[c2l_obs["sample"].astype(str) == SAMPLE_TO_C2L[sample], list(C2L_TYPES)]
    c2l = sub.reindex(spots["barcode"]).astype(float)
    c2l.index = range(len(c2l))
    keep = c2l.notna().all(axis=1) & (c2l.sum(axis=1) > 0) & (h2p_c2l["n_nuclei"] > 0)
    frac_h = h2p_c2l[[f"frac_{g}" for g in C2L_TYPES]].copy(); frac_h.columns = list(C2L_TYPES)
    frac_h = vv.proportions(frac_h.clip(lower=0))      # renormalise over the 7 c2l types
    cnt_h = h2p_c2l[[f"count_{g}" for g in C2L_TYPES]].copy(); cnt_h.columns = list(C2L_TYPES)
    prop_tab = vv.agreement(frac_h[keep], vv.proportions(c2l)[keep]).assign(unit="proportion")
    cnt_tab = vv.agreement(cnt_h[keep], c2l[keep]).assign(unit="count")
    slide = pd.DataFrame({
        "Hist2Pheno argmax": pd.Series(npc_figures.map_l2(class_names[i] for i in probs.argmax(1))).value_counts(normalize=True).reindex(C2L_TYPES).fillna(0),
        "Hist2Pheno mean softmax": vv.cell_fractions(cnt_h[keep]),
        "cell2location (sum of abundance)": vv.cell_fractions(c2l[keep]),
    })
    print(f"\n===== {sample}: spots {len(spots):,}, capture radius {radius:.1f} px (55 µm), "
          f"spots with nuclei and c2l {int(keep.sum()):,}, nuclei/spot median {h2p_c2l.n_nuclei[keep].median():.0f}")
    print("slide-level cell fractions (c2l by abundance, not argmax)")
    display(slide.round(4))
    print("per-spot agreement with cell2location")
    display(pd.concat([prop_tab, cnt_tab]).pivot_table(index="h2p", columns="unit", values="spearman").reindex(C2L_TYPES).round(3))

    # (b) RNA markers, if the count matrix is present
    h5 = visium_dir / "filtered_feature_bc_matrix.h5"
    mk_tab = None
    if h5.is_file():
        h2p = vv.spot_h2p(xy, probs, class_names, spots, radius)
        ms = vv.marker_scores(h5, spots["barcode"], MARKER_SETS)
        ms.index = range(len(ms))
        mk_tab = vv.agreement(h2p, ms, MARKER_PAIRS)
        c2l_mk = vv.agreement(vv.proportions(c2l).rename(columns={"B": "B_c2l", "T": "T_c2l", "Myeloid": "Myeloid_c2l"}),
                              ms, {"B_c2l": "B", "T_c2l": "T", "Myeloid_c2l": "Myeloid"})
        print("per-spot agreement with RNA marker scores (genes used:", ms.attrs["genes_used"], ")")
        display(mk_tab.round(3))
        print("reference: cell2location proportion vs the same markers")
        display(c2l_mk.round(3))
        mk_tab.to_csv(fig_dir / f"{sample}_h2p_vs_markers.csv", index=False)
    else:
        print(f"[markers] {h5} not found; copy the Space Ranger filtered_feature_bc_matrix.h5 there to enable the RNA check")
    slide.to_csv(fig_dir / f"{sample}_cell_fractions_h2p_vs_c2l.csv")
    pd.concat([prop_tab, cnt_tab]).to_csv(fig_dir / f"{sample}_h2p_vs_c2l_spot_agreement.csv", index=False)
    results[sample] = {"slide": slide, "c2l": pd.concat([prop_tab, cnt_tab]), "markers": mk_tab}

### How to read this

* If **Tumor/Epithelial** agree with the RNA markers (Spearman > ~0.3) but **B/T** do not, the slide is aligned and the immune error is the model. If Tumor also fails, check the HE/Visium registration first (`check_visium_alignment.py`).
* `count` agreement is usually higher than `proportion` agreement because both scale with how many cells sit in a spot; `proportion` is the stricter test.
* cell2location vs markers is a ceiling: c2l was fitted on the same RNA.